In [ ]:
import os
import pandas as pd
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from dtaidistance import dtw
import csv

In [ ]:
#prediction versus test
def calculate_similarity_dtw_components(path, column_test, column_predict, freq=4):
    df = pd.read_csv(path)
    Test= df[column_test]
    Prediction= df[column_predict]

    decomposition_Test = seasonal_decompose(Test, model='additive', period=freq)
    decomposition_Prediction = seasonal_decompose(Prediction, model='additive', period=freq)
    
    trend_Test = decomposition_Test.trend.dropna().values.astype(np.float64)
    trend_Prediction = decomposition_Prediction.trend.dropna().values.astype(np.float64)
    seasonal_Test = decomposition_Test.seasonal.dropna().values.astype(np.float64)
    seasonal_Prediction = decomposition_Prediction.seasonal.dropna().values.astype(np.float64)
    
    min_length = min(len(trend_Test), len(trend_Prediction), len(seasonal_Test), len(seasonal_Prediction),
                     len(Test), len(Prediction))
    
    trend_Test = trend_Test[:min_length]
    trend_Prediction = trend_Prediction[:min_length]
    seasonal_Test = seasonal_Test[:min_length]
    seasonal_Prediction = seasonal_Prediction[:min_length]
    Test = Test[:min_length].values.astype(np.float64)
    Prediction = Prediction[:min_length].values.astype(np.float64)
    #plotar
    # plt.plot(Test, label = 'Teste')
    # plt.plot(Prediction)
    # plt.show()
    # plt.plot(trend_Test, label = 'Teste')
    # plt.plot(trend_Prediction)
    # plt.show()
    # plt.plot(seasonal_Test, label = 'Teste')
    # plt.plot(seasonal_Prediction)
    # plt.show()
    
    trend_distance = dtw.distance(trend_Test, trend_Prediction)
    seasonal_distance = dtw.distance(seasonal_Test, seasonal_Prediction)
    observation_distance = dtw.distance(Test, Prediction)

    max = df['Prediction'].max()

    return trend_distance/max,observation_distance/max,seasonal_distance/max


In [ ]:
#dtw entre o dataset original e a predição 

folder_path = '../results/bi-lstm/forecast'
output_csv = '../results/DTW_LSTM_GRU.csv'

with open(output_csv, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['link', 'tcp', 'model', 'imputation', 
                     'dtw_seasonal', 'dtw_trend', 'dtw_observation'])

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('.csv'):
            try:
                file_path = os.path.join(root, file)
                file_name = os.path.basename(file_path).split('.')[0]    
                
                link = file_name.split(' ')[8]
                tcp = file_name.split(' ')[5]
                model = file_name.split(' ')[1]
                imputation = file_name.split(' ')[2]
                
                dtw_results = calculate_similarity_dtw_components(
                    file_path, 'Test', 'Prediction', freq=4
                )
                
                if isinstance(dtw_results, tuple):
                    trend, observation, seasonal = dtw_results
                elif isinstance(dtw_results, dict):
                    trend = dtw_results['trend_dtw_distance']
                    observation = dtw_results['observation_dtw_distance']
                    seasonal = dtw_results['seasonal_dtw_distance']
                else:
                    raise ValueError("Formato inesperado retornado por calculate_similarity_dtw_components")
                
                with open(output_csv, mode='a', newline='', encoding='utf-8') as csvfile:
                    writer = csv.writer(csvfile)
                    writer.writerow([link, tcp, model, imputation, 
                                     seasonal, trend, observation])
                
                print(f"Resultados salvos para {file_name}")

            except IndexError:
                print(f"Erro: 'list index out of range' no arquivo {file_name}. Pulando para o próximo arquivo.")
            
            except Exception as e:
                print(f"Erro inesperado no arquivo {file_name}: {e}. Pulando para o próximo arquivo.")




In [ ]:
#imputed versus original
def calculate_similarity_dtw_components_imputed(path_original, column_original, path_imputed, column_imputed, freq=4):
    df_original = pd.read_csv(path_original)
    original= df_original[column_original]
    df_imputed = pd.read_csv(path_imputed)
    imputed= df_imputed[column_imputed]

    decomposition_original = seasonal_decompose(original, model='additive', period=freq)
    decomposition_imputed = seasonal_decompose(imputed, model='additive', period=freq)
    
    trend_original = decomposition_original.trend.dropna().values.astype(np.float64)
    trend_imputed = decomposition_imputed.trend.dropna().values.astype(np.float64)
    seasonal_original = decomposition_original.seasonal.dropna().values.astype(np.float64)
    seasonal_imputed = decomposition_imputed.seasonal.dropna().values.astype(np.float64)
    
    min_length = min(len(trend_original), len(trend_imputed), len(seasonal_original), len(seasonal_imputed),
                     len(original), len(imputed))
    
    trend_original = trend_original[:min_length]
    trend_imputed = trend_imputed[:min_length]
    seasonal_original = seasonal_original[:min_length]
    seasonal_imputed = seasonal_imputed[:min_length]
    original = original[:min_length].values.astype(np.float64)
    imputed = imputed[:min_length].values.astype(np.float64)
    #plotar
    # plt.plot(original, label = 'original')
    # plt.plot(imputed)
    # plt.show()
    # plt.plot(trend_original, label = 'original')
    # plt.plot(trend_imputed)
    # plt.show()
    # plt.plot(seasonal_original, label = 'original')
    # plt.plot(seasonal_imputed)
    # plt.show()
    
    trend_distance = dtw.distance(trend_original, trend_imputed)
    seasonal_distance = dtw.distance(seasonal_original, seasonal_imputed)
    observation_distance = dtw.distance(original, imputed)

    max = imputed.max()

    return trend_distance/max,observation_distance/max,seasonal_distance/max


In [ ]:
trend, obs, seas = calculate_similarity_dtw_components_imputed('../datasets/treated-longest-interval/treated bbr esmond data ap-ba 07-03-2023_longest_interval.csv', 'Throughput','../datasets/imputed-treated-longest-interval-with-failures/interpolacao-linear/treated bbr esmond data ap-ba 07-03-2023_longest_interval.csv','Throughput', freq=4)


In [ ]:
trend, obs, seas

In [ ]:
import os
import csv

# Caminhos das pastas
imputation_path = '../datasets/imputed-treated-longest-interval-with-failures/' 
original_path = '../datasets/treated-longest-interval/' 
output_csv = '../results/DTW_imputation_original.csv'

# Lista de arquivos presentes na pasta original
original_files = {}
for root, dirs, files in os.walk(original_path):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            file_name = os.path.basename(file_path).split('.')[0]
            original_files[file_name] = file_path  # Mapeia o nome base do arquivo para o caminho completo

# Criação do CSV de saída
with open(output_csv, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['link', 'tcp', 'imputation', 'dtw_seasonal', 'dtw_trend', 'dtw_observation'])

# Processamento dos arquivos imputados
for root, dirs, files in os.walk(imputation_path):
    for file in files:
        if file.endswith('.csv'):
            try:
                # Obter informações do arquivo imputado
                file_path = os.path.join(root, file)
                file_name = os.path.basename(file_path).split('.')[0]
                
                link = file_name.split(' ')[4]
                tcp = file_name.split(' ')[1]
                imputation = root.split('/')[-1]

                # Verificar se o arquivo correspondente está na pasta original
                if file_name in original_files:
                    file_path2 = original_files[file_name]  # Caminho do arquivo original correspondente

                    # Calcular DTW
                    dtw_results = calculate_similarity_dtw_components_imputed(
                        file_path2, 'Throughput', file_path, 'Throughput', freq=4
                    )

                    if isinstance(dtw_results, tuple):
                        trend, observation, seasonal = dtw_results
                    elif isinstance(dtw_results, dict):
                        trend = dtw_results['trend_dtw_distance']
                        observation = dtw_results['observation_dtw_distance']
                        seasonal = dtw_results['seasonal_dtw_distance']
                    else:
                        raise ValueError("Formato inesperado retornado por calculate_similarity_dtw_components")

                    # Salvar resultados no CSV
                    with open(output_csv, mode='a', newline='', encoding='utf-8') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow([link, tcp, imputation, seasonal, trend, observation])

                    print(f"Resultados salvos para {file_name}")
                else:
                    print(f"Arquivo correspondente para {file_name} não encontrado na pasta original. Pulando...")

            except IndexError:
                print(f"Erro: 'list index out of range' no arquivo {file_name}. Pulando para o próximo arquivo.")

            except Exception as e:
                print(f"Erro inesperado no arquivo {file_name}: {e}. Pulando para o próximo arquivo.")
